In [1]:
# %% [markdown]
# # Arabic Benchmark Harness (Al-Ghafa + Ashaar)
#
# - Tasks:
#   1) AlGhafa-ARC (MCQ)
#   2) AlGhafa-MMLU (MCQ)
#   3) AlGhafa-BoolQ (نعم/لا)
#   4) AlGhafa-PiQA (2-way MCQ)
#   5) Ashaar – rewrite bayt with full diacritics
#   6) Ashaar – theme classification
#   7) Ashaar – era classification
#
# - One Arabic prompt template **per task type**.
# - Decode modes: `det` (greedy) and `samp` (sampling).
# - This notebook first runs a **tiny "medium" sanity run (5 examples/task)**.
# - It runs the suite for two models:
#     - Hala-9B
#     - Yehia-7B-preview


In [2]:
# %%
# ============================================================
# 🔧 SAFE ENVIRONMENT SETUP FOR RUNPOD / HF MODELS
# ============================================================

# Disable HF fast-transfer (causes ValueError if hf_transfer not installed)
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
print("✓ Disabled HF_HUB_ENABLE_HF_TRANSFER")

# Install required packages
!pip install -q accelerate hf_transfer transformers datasets numpy pandas torch tqdm sentencepiece

# Core imports
import os
import re
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

print("✓ Environment ready. If this was the first run: RESTART kernel now.")


✓ Disabled HF_HUB_ENABLE_HF_TRANSFER
✓ Environment ready. If this was the first run: RESTART kernel now.


In [3]:
# %%
# =========================
# Global config
# =========================

# Models to test
HALA_ID  = "hammh0a/Hala-9B"
YEHIA_ID = "Navid-AI/Yehia-7B-preview"

# Where Al-Ghafa repo lives (adjust if needed)
AL_GHAFA_ROOT = "alghafa/arabic-eval"

# Output dirs
MEDIUM_OUT_DIR = Path("results_medium")
FULL_OUT_DIR = Path("results_full")
MEDIUM_OUT_DIR.mkdir(exist_ok=True)
FULL_OUT_DIR.mkdir(exist_ok=True)

# Decode modes we support
DECODE_MODES = ["det", "samp"]

# "Medium" sanity sample sizes (tiny, just to check that code is coding 😄)
MEDIUM_SIZES = {
    "alghafa_arc": 5,
    "alghafa_mmlu": 5,
    "alghafa_boolq": 5,
    "alghafa_piqa": 5,
    "ashaar_rewrite": 5,
    "ashaar_theme": 5,
    "ashaar_era": 5,
}

# Suggested "full" sizes (you can tweak later)
FULL_SIZES = {
    "alghafa_arc": 300,   # None == use full dataset
    "alghafa_mmlu": 300,
    "alghafa_boolq": 300,
    "alghafa_piqa": 300,
    "ashaar_rewrite": 300,
    "ashaar_theme": 300,
    "ashaar_era": 300,
}

RANDOM_SEED = 1337


In [4]:
# %%
# Small helper for seeding

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)


In [5]:
# %%
# =========================
# Model loading
# =========================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

def fix_pad_eos(tok, model):
    # Ensure PAD/EOS hooked to avoid warnings and odd stops
    if tok.pad_token is None:
        if tok.eos_token is not None:
            tok.pad_token = tok.eos_token
        else:
            tok.add_special_tokens({"pad_token": "[PAD]"})
            model.resize_token_embeddings(len(tok))
    model.config.pad_token_id = tok.pad_token_id
    if model.config.eos_token_id is None and tok.eos_token_id is not None:
        model.config.eos_token_id = tok.eos_token_id
    if hasattr(model, "generation_config"):
        model.generation_config.pad_token_id = tok.pad_token_id
        if tok.eos_token_id is not None:
            model.generation_config.eos_token_id = tok.eos_token_id

# Global handles that will be swapped when we change models
tokenizer = None
model = None
MODEL_TAG = ""

def load_model(model_id):
    """Load a model into the global `model`, `tokenizer`, `MODEL_TAG`."""
    global tokenizer, model, MODEL_TAG

    print("\n" + "="*60)
    print("Loading model:", model_id)
    print("="*60)

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    fix_pad_eos(tokenizer, model)

    MODEL_TAG = model_id.split("/")[-1].replace("/", "-").replace(".", "_")
    print("Model tag:", MODEL_TAG)


Using device: cuda


In [6]:
# %%
# =========================
# Decode helper
# =========================

def generate_text(prompt, decode_mode="det", max_new_tokens=64):
    """Generate short text given a prompt and decode mode (uses global model)."""
    if tokenizer is None or model is None:
        raise RuntimeError("Call load_model(model_id) before generation.")

    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    if decode_mode == "det":
        gen = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    elif decode_mode == "samp":
        gen = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    else:
        raise ValueError(f"Unknown decode_mode: {decode_mode}")

    # Take only the new tokens
    generated = gen[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True)
    return text.strip()


In [7]:
# %%
# =========================
# Load Al-Ghafa datasets
# =========================

AL_GHAFA_PATHS = {
    "alghafa_arc":   f"{AL_GHAFA_ROOT}/arc_challenge_okapi_ar/eval/arc_challenge_ar_okapi_test.csv",
    "alghafa_mmlu":  f"{AL_GHAFA_ROOT}/mmlu_okapi_ar/eval/mmlu_ar_okapi_test.csv",
    "alghafa_boolq": f"{AL_GHAFA_ROOT}/boolq_ar/eval/translated_boolq_valid.csv",
    "alghafa_piqa":  f"{AL_GHAFA_ROOT}/piqa_ar/eval/translated_piqa_valid.csv",
}

alghafa_dfs = {}
for key, path in AL_GHAFA_PATHS.items():
    df = pd.read_csv(path)
    alghafa_dfs[key] = df
    print(f"{key}: loaded {df.shape[0]} rows from {path}")


alghafa_arc: loaded 1169 rows from alghafa/arabic-eval/arc_challenge_okapi_ar/eval/arc_challenge_ar_okapi_test.csv
alghafa_mmlu: loaded 12928 rows from alghafa/arabic-eval/mmlu_okapi_ar/eval/mmlu_ar_okapi_test.csv
alghafa_boolq: loaded 3270 rows from alghafa/arabic-eval/boolq_ar/eval/translated_boolq_valid.csv
alghafa_piqa: loaded 1838 rows from alghafa/arabic-eval/piqa_ar/eval/translated_piqa_valid.csv


In [8]:
# %%
# =========================
# Load Ashaar dataset
# =========================
!pip install hf_transfer
ashaar_raw = load_dataset("arbml/ashaar")
ashaar_split_name = list(ashaar_raw.keys())[0]  # e.g. 'train'
ashaar_ds = ashaar_raw[ashaar_split_name]
print("Ashaar split:", ashaar_split_name, "rows:", len(ashaar_ds))

# --- Collect label sets for theme / era, skipping None/empty ---
theme_values = []
theme_valid_indices = []

era_values = []
era_valid_indices = []

for i in range(len(ashaar_ds)):
    theme = ashaar_ds[i]["poem theme"]
    era   = ashaar_ds[i]["poet era"]

    if isinstance(theme, str) and theme.strip():
        theme_values.append(theme)
        theme_valid_indices.append(i)

    if isinstance(era, str) and era.strip():
        era_values.append(era)
        era_valid_indices.append(i)

theme_labels = sorted(set(theme_values))
era_labels   = sorted(set(era_values))

print("Number of unique themes (non-null):", len(theme_labels))
print("Number of unique eras   (non-null):", len(era_labels))
print("Valid theme-labelled examples:", len(theme_valid_indices))
print("Valid era-labelled examples:  ", len(era_valid_indices))


Ashaar split: train rows: 254630
Number of unique themes (non-null): 18
Number of unique eras   (non-null): 14
Valid theme-labelled examples: 67520
Valid era-labelled examples:   147421


In [9]:
# %%
# =========================
# Prompt builders (Arabic)
# =========================

def build_mcq_prompt_ar(question_text, options, task_name="MCQ"):
    """Generic Arabic MCQ prompt (4 options)."""
    letters = ["أ", "ب", "ج", "د"]
    opts_lines = []
    for i, opt in enumerate(options):
        if i >= len(letters):
            break
        opts_lines.append(f"{letters[i]}) {opt}")
    opts_str = "\n".join(opts_lines)

    prompt = (
        "أنت نموذج لغة عربي يساعد في حل أسئلة اختيار من متعدد.\n"
        "اقرأ السؤال والخيارات، ثم أجب بحرف الخيار الصحيح فقط "
        "(أ أو ب أو ج أو د)، بدون أي شرح إضافي.\n\n"
        f"السؤال ({task_name}):\n{question_text}\n\n"
        f"الخيارات:\n{opts_str}\n\n"
        "الإجابة (اكتب حرفًا واحدًا فقط):"
    )
    return prompt

def build_boolq_prompt_ar(passage, question_text):
    prompt = (
        "أنت نموذج يجيب عن أسئلة نعم/لا باللغة العربية.\n"
        "باستخدام المقطع التالي، أجب عن السؤال بكلمة واحدة فقط: نعم أو لا.\n\n"
        f"المقطع:\n{passage}\n\n"
        f"السؤال:\n{question_text}\n\n"
        "الإجابة (نعم أو لا فقط):"
    )
    return prompt

def build_piqa_prompt_ar(goal, options):
    # options = [sol1, sol2]
    letters = ["أ", "ب"]
    opts_lines = [f"{letters[i]}) {opt}" for i, opt in enumerate(options)]
    opts_str = "\n".join(opts_lines)
    prompt = (
        "أنت نموذج يفهم المعرفة العملية والحس السليم.\n"
        "سأعطيك هدفًا، وخيارين محتملين لتحقيق هذا الهدف.\n"
        "اختر الخيار الأكثر منطقية عن طريق كتابة حرف واحد فقط (أ أو ب).\n\n"
        f"الهدف:\n{goal}\n\n"
        f"الخيارات:\n{opts_str}\n\n"
        "الإجابة (حرف واحد فقط):"
    )
    return prompt

def build_ashaar_rewrite_prompt_ar(bayt):
    prompt = (
        "أنت نموذج يولّد الشعر العربي.\n"
        "سأعطيك بيتَ شعرٍ عربيًّا مشكولًا بالكامل.\n"
        "مهمتك الوحيدة هي إعادة كتابة البيت نفسه تمامًا، "
        "بنفس الكلمات ونفس التشكيل، دون أي تعديل أو إضافة أو شرح.\n\n"
        f"البيت:\n{bayt}\n\n"
        "أعد كتابة البيت كما هو تمامًا:"
    )
    return prompt

def build_ashaar_label_prompt_ar(poem_text, labels, label_type="الموضوع"):
    labels_str = "، ".join(labels)
    prompt = (
        "أنت نموذج يفهم مواضيع وفترات الشعر العربي.\n"
        "سأعطيك مقطعًا من قصيدة، ومهمتك اختيار "
        f"{label_type} المناسب من القائمة المعطاة.\n"
        "اختر اسمًا واحدًا فقط من القائمة واكتبه كما هو بالضبط، بدون شرح.\n\n"
        f"المقطع:\n{poem_text}\n\n"
        f"{label_type}ات الممكنة:\n{labels_str}\n\n"
        f"{label_type} القصيدة هو:"
    )
    return prompt


In [10]:
# %%
# =========================
# Parsing helpers
# =========================

def extract_mcq_index_from_text(text, num_options=4):
    """Map first reasonable symbol in the answer to option index (0..num_options-1)."""
    text = text.strip()
    arabic_map = {"أ": 0, "ا": 0, "ب": 1, "ج": 2, "د": 3}
    latin_map = {"A": 0, "B": 1, "C": 2, "D": 3}
    digit_map  = {"1": 0, "2": 1, "3": 2, "4": 3}

    for ch in text:
        if ch in arabic_map and arabic_map[ch] < num_options:
            return arabic_map[ch]
        if ch in latin_map and latin_map[ch] < num_options:
            return latin_map[ch]
        if ch in digit_map and digit_map[ch] < num_options:
            return digit_map[ch]
    return None

def parse_boolq_answer(text):
    """Return True/False/None based on Arabic yes/no."""
    t = text.strip()
    # squash spaces and punctuation a bit
    t = re.sub(r"[\s\.\،\؛\:\!]+", "", t)
    if t.startswith("نعم"):
        return True
    if t.startswith("لا"):
        return False
    return None

def normalize_label_text(text):
    if text is None:
        return ""
    t = str(text)
    t = t.strip()
    t = re.sub(r"[\s\.\،\؛\:\!]+", " ", t)
    return t

def pick_label_from_prediction(pred_text, label_list):
    """Try to map model text to one of label_list (by substring / equality)."""
    norm_pred = normalize_label_text(pred_text)
    if not norm_pred:
        return None

    # First try exact normalized match
    for lab in label_list:
        if normalize_label_text(lab) == norm_pred:
            return lab

    # Then try "label is substring of prediction"
    for lab in label_list:
        if normalize_label_text(lab) in norm_pred:
            return lab

    # Then try "prediction is substring of label"
    for lab in label_list:
        if norm_pred in normalize_label_text(lab):
            return lab

    return None

def char_match_ratio(target, pred):
    """Very simple character-level overlap metric."""
    if not target:
        return 1.0 if not pred else 0.0
    L = max(len(target), 1)
    matches = sum(1 for a, b in zip(target, pred) if a == b)
    return matches / L


In [11]:
# %%
# =========================
# Utility: sample indices
# =========================

def sample_indices(n_total, n_sample, seed, replace=False):
    n_sample = min(n_sample, n_total)
    rng = random.Random(seed)
    if replace or n_sample > n_total:
        return [rng.randrange(0, n_total) for _ in range(n_sample)]
    return rng.sample(range(n_total), n_sample)


In [12]:
# %%
# =========================
# Al-Ghafa task runners
# =========================

def run_alghafa_arc(df, n_sample, decode_mode, out_dir, mode_tag="medium"):
    """ARC-Challenge (4-way MCQ)."""
    n_total = df.shape[0]
    idxs = sample_indices(n_total, n_sample, RANDOM_SEED)
    rows = []

    for i in tqdm(idxs, desc=f"ARC ({decode_mode}, {mode_tag})"):
        row = df.iloc[i]
        q = str(row["query"])
        options = [str(row[f"sol{j}"]) for j in range(1, 5)]
        gold_idx = int(row["label"])

        prompt = build_mcq_prompt_ar(q, options, task_name="ARC")
        pred_text = generate_text(prompt, decode_mode=decode_mode, max_new_tokens=8)
        pred_idx = extract_mcq_index_from_text(pred_text, num_options=4)

        correct = (pred_idx == gold_idx)
        rows.append({
            "row_idx": i,
            "question": q,
            "sol1": options[0],
            "sol2": options[1],
            "sol3": options[2],
            "sol4": options[3],
            "gold_label_idx": gold_idx,
            "pred_text": pred_text,
            "pred_label_idx": pred_idx,
            "correct": int(bool(correct)),
        })

    df_out = pd.DataFrame(rows)
    acc = df_out["correct"].mean() if len(df_out) > 0 else 0.0
    print(f"ARC {mode_tag} ({decode_mode}) accuracy: {acc:.3f}")

    fname = out_dir / f"alghafa_arc_{mode_tag}_{MODEL_TAG}_{decode_mode}.csv"
    df_out.to_csv(fname, index=False, encoding="utf-8")
    print("Saved:", fname)
    return df_out, acc


def run_alghafa_mmlu(df, n_sample, decode_mode, out_dir, mode_tag="medium"):
    """MMLU-Ar (4-way MCQ)."""
    n_total = df.shape[0]
    idxs = sample_indices(n_total, n_sample, RANDOM_SEED + 1)
    rows = []

    for i in tqdm(idxs, desc=f"MMLU ({decode_mode}, {mode_tag})"):
        row = df.iloc[i]
        q = str(row["question"])
        options = [str(row[f"sol{j}"]) for j in range(1, 5)]
        gold_idx = int(row["label"])

        prompt = build_mcq_prompt_ar(q, options, task_name="MMLU-Ar")
        pred_text = generate_text(prompt, decode_mode=decode_mode, max_new_tokens=8)
        pred_idx = extract_mcq_index_from_text(pred_text, num_options=4)

        correct = (pred_idx == gold_idx)
        rows.append({
            "row_idx": i,
            "question": q,
            "sol1": options[0],
            "sol2": options[1],
            "sol3": options[2],
            "sol4": options[3],
            "gold_label_idx": gold_idx,
            "pred_text": pred_text,
            "pred_label_idx": pred_idx,
            "correct": int(bool(correct)),
        })

    df_out = pd.DataFrame(rows)
    acc = df_out["correct"].mean() if len(df_out) > 0 else 0.0
    print(f"MMLU-Ar {mode_tag} ({decode_mode}) accuracy: {acc:.3f}")

    fname = out_dir / f"alghafa_mmlu_{mode_tag}_{MODEL_TAG}_{decode_mode}.csv"
    df_out.to_csv(fname, index=False, encoding="utf-8")
    print("Saved:", fname)
    return df_out, acc


def run_alghafa_boolq(df, n_sample, decode_mode, out_dir, mode_tag="medium"):
    """BoolQ-Ar (نعم/لا)."""
    n_total = df.shape[0]
    idxs = sample_indices(n_total, n_sample, RANDOM_SEED + 2)
    rows = []

    for i in tqdm(idxs, desc=f"BoolQ ({decode_mode}, {mode_tag})"):
        row = df.iloc[i]
        q = str(row["question"])
        passage = str(row["passage"])
        gold_raw = str(row["answer"]).strip().lower()
        gold_bool = True if gold_raw in ["true", "1", "yes"] else False

        prompt = build_boolq_prompt_ar(passage, q)
        pred_text = generate_text(prompt, decode_mode=decode_mode, max_new_tokens=5)
        pred_bool = parse_boolq_answer(pred_text)

        correct = (pred_bool is not None and pred_bool == gold_bool)
        rows.append({
            "row_idx": i,
            "question": q,
            "passage": passage,
            "gold_answer_bool": int(gold_bool),
            "pred_text": pred_text,
            "pred_answer_bool": None if pred_bool is None else int(pred_bool),
            "correct": int(bool(correct)),
        })

    df_out = pd.DataFrame(rows)
    acc = df_out["correct"].mean() if len(df_out) > 0 else 0.0
    print(f"BoolQ-Ar {mode_tag} ({decode_mode}) accuracy: {acc:.3f}")

    fname = out_dir / f"alghafa_boolq_{mode_tag}_{MODEL_TAG}_{decode_mode}.csv"
    df_out.to_csv(fname, index=False, encoding="utf-8")
    print("Saved:", fname)
    return df_out, acc


def run_alghafa_piqa(df, n_sample, decode_mode, out_dir, mode_tag="medium"):
    """PiQA-Ar (2-way MCQ)."""
    n_total = df.shape[0]
    idxs = sample_indices(n_total, n_sample, RANDOM_SEED + 3)
    rows = []

    for i in tqdm(idxs, desc=f"PiQA ({decode_mode}, {mode_tag})"):
        row = df.iloc[i]
        goal = str(row["goal"]) if "goal" in row else ""
        sol1 = str(row["sol1"])
        sol2 = str(row["sol2"])
        options = [sol1, sol2]
        gold_idx = int(row["label"])

        prompt = build_piqa_prompt_ar(goal, options)
        pred_text = generate_text(prompt, decode_mode=decode_mode, max_new_tokens=8)
        pred_idx = extract_mcq_index_from_text(pred_text, num_options=2)

        correct = (pred_idx == gold_idx)
        rows.append({
            "row_idx": i,
            "goal": goal,
            "sol1": sol1,
            "sol2": sol2,
            "gold_label_idx": gold_idx,
            "pred_text": pred_text,
            "pred_label_idx": pred_idx,
            "correct": int(bool(correct)),
        })

    df_out = pd.DataFrame(rows)
    acc = df_out["correct"].mean() if len(df_out) > 0 else 0.0
    print(f"PiQA-Ar {mode_tag} ({decode_mode}) accuracy: {acc:.3f}")

    fname = out_dir / f"alghafa_piqa_{mode_tag}_{MODEL_TAG}_{decode_mode}.csv"
    df_out.to_csv(fname, index=False, encoding="utf-8")
    print("Saved:", fname)
    return df_out, acc


In [13]:
# %%
# ---------------------------------
# Helper: normalize Ashaar themes
# ---------------------------------

# map common free-form outputs → canonical labels from the dataset
THEME_SYNONYMS = {
    "قصيدة غزل": "قصيدة رومنسيه",
    "قصيدة غزلية": "قصيدة رومنسيه",
    "الغزل": "قصيدة رومنسيه",
    "قصيدة اعتذار": "قصيدة عتاب",
    # add more if you see new variants
}

def normalize_theme_label_from_text(pred_text, label_list):
    """Turn raw model output into one of the known theme labels, if possible."""
    if pred_text is None:
        return None

    text = str(pred_text)

    # strip punctuation / tatweel / newlines, collapse spaces
    for ch in ["\n", "،", ".", "!", "؟", "ـ"]:
        text = text.replace(ch, " ")
    text = " ".join(text.split())

    # 1) direct hit: canonical label appears inside the text
    for lab in label_list:
        if lab and lab in text:
            return lab

    # 2) synonyms: map variant → canonical
    for syn, target in THEME_SYNONYMS.items():
        if syn in text and target in label_list:
            return target

    # 3) last resort: exact match after cleaning
    if text in label_list:
        return text

    return None


In [14]:
# %%
# =========================
# Ashaar task runners
# =========================

def get_ashaar_example(idx):
    ex = ashaar_ds[int(idx)]
    return ex

def extract_poem_text(ex, max_verses=2):
    verses = ex.get("poem verses", "")
    if isinstance(verses, list):
        return "\n".join(verses[:max_verses])
    return str(verses)

def extract_bayt_for_rewrite(ex):
    verses = ex.get("poem verses", "")
    if isinstance(verses, list) and len(verses) > 0:
        return verses[0]
    return str(verses)

def run_ashaar_rewrite(n_sample, decode_mode, out_dir, mode_tag="medium"):
    n_total = len(ashaar_ds)
    idxs = sample_indices(n_total, n_sample, RANDOM_SEED + 4)
    rows = []

    for i in tqdm(idxs, desc=f"Ashaar rewrite ({decode_mode}, {mode_tag})"):
        ex = get_ashaar_example(i)
        bayt = extract_bayt_for_rewrite(ex)
        prompt = build_ashaar_rewrite_prompt_ar(bayt)
        pred_text = generate_text(
            prompt,
            decode_mode=decode_mode,
            max_new_tokens=len(bayt) + 10,
        )

        em = int(pred_text.strip() == bayt.strip())
        ratio = char_match_ratio(bayt, pred_text)

        rows.append({
            "row_idx": i,
            "poem_title": ex.get("poem title", ""),
            "poet_name": ex.get("poet name", ""),
            "gold_bayt": bayt,
            "pred_bayt": pred_text,
            "exact_match": em,
            "char_match_ratio": ratio,
        })

    df_out = pd.DataFrame(rows)
    em_mean = df_out["exact_match"].mean() if len(df_out) > 0 else 0.0
    ratio_mean = df_out["char_match_ratio"].mean() if len(df_out) > 0 else 0.0
    print(f"Ashaar rewrite {mode_tag} ({decode_mode}) | EM={em_mean:.3f}, char_match={ratio_mean:.3f}")

    fname = out_dir / f"ashaar_rewrite_{mode_tag}_{MODEL_TAG}_{decode_mode}.csv"
    df_out.to_csv(fname, index=False, encoding="utf-8")
    print("Saved:", fname)
    return df_out, em_mean, ratio_mean


def run_ashaar_label_task(
    n_sample,
    decode_mode,
    out_dir,
    label_list,
    label_key,
    mode_tag="medium",
    valid_indices=None,
):
    """
    Generic Ashaar label task:
      label_key in {"poem theme", "poet era"}.

    If valid_indices is given, we only sample from those indices
    (where the label is non-null).
    """
    if valid_indices is not None:
        base_indices = list(valid_indices)
    else:
        base_indices = list(range(len(ashaar_ds)))

    n_total = len(base_indices)

    seed_offset = 5 if label_key == "poem theme" else 6
    sampled_positions = sample_indices(n_total, n_sample, RANDOM_SEED + seed_offset)
    idxs = [base_indices[pos] for pos in sampled_positions]

    rows = []

    label_type = "الموضوع" if label_key == "poem theme" else "العصر"

    for ds_idx in tqdm(idxs, desc=f"Ashaar {label_type} ({decode_mode}, {mode_tag})"):
        ex = get_ashaar_example(ds_idx)
        poem_text = extract_poem_text(ex, max_verses=3)
        gold_label = ex.get(label_key, "")

        prompt = build_ashaar_label_prompt_ar(poem_text, label_list, label_type=label_type)
        pred_text = generate_text(prompt, decode_mode=decode_mode, max_new_tokens=16)

        # 🔧 key change: for themes, use the normalizer + fallback;
        # for eras, keep old behaviour.
        if label_key == "poem theme":
            pred_label = normalize_theme_label_from_text(pred_text, label_list)
            if pred_label is None:
                pred_label = pick_label_from_prediction(pred_text, label_list)
        else:
            pred_label = pick_label_from_prediction(pred_text, label_list)

        correct = (pred_label == gold_label)
        rows.append({
            "row_idx": ds_idx,
            "poem_title": ex.get("poem title", ""),
            "poet_name": ex.get("poet name", ""),
            "poem_text_snippet": poem_text,
            "gold_label": gold_label,
            "pred_text": pred_text,
            "pred_label": pred_label,
            "correct": int(bool(correct)),
        })

    df_out = pd.DataFrame(rows)
    acc = df_out["correct"].mean() if len(df_out) > 0 else 0.0
    short_key = "theme" if label_key == "poem theme" else "era"
    print(f"Ashaar {short_key} {mode_tag} ({decode_mode}) accuracy: {acc:.3f}")

    fname = out_dir / f"ashaar_{short_key}_{mode_tag}_{MODEL_TAG}_{decode_mode}.csv"
    df_out.to_csv(fname, index=False, encoding="utf-8")
    print("Saved:", fname)
    return df_out, acc


In [15]:
# %%
# =========================
#  MEDIUM sanity run (5 examples each) for CURRENT model
# =========================

def run_medium_suite_for_decode(decode_mode):
    print("\n" + "="*60)
    print(f"Running MEDIUM suite | model = {MODEL_TAG} | decode_mode = {decode_mode}")
    print("="*60)

    # Al-Ghafa tasks
    run_alghafa_arc(
        alghafa_dfs["alghafa_arc"],
        MEDIUM_SIZES["alghafa_arc"],
        decode_mode,
        MEDIUM_OUT_DIR,
        mode_tag="medium",
    )

    run_alghafa_mmlu(
        alghafa_dfs["alghafa_mmlu"],
        MEDIUM_SIZES["alghafa_mmlu"],
        decode_mode,
        MEDIUM_OUT_DIR,
        mode_tag="medium",
    )

    run_alghafa_boolq(
        alghafa_dfs["alghafa_boolq"],
        MEDIUM_SIZES["alghafa_boolq"],
        decode_mode,
        MEDIUM_OUT_DIR,
        mode_tag="medium",
    )

    run_alghafa_piqa(
        alghafa_dfs["alghafa_piqa"],
        MEDIUM_SIZES["alghafa_piqa"],
        decode_mode,
        MEDIUM_OUT_DIR,
        mode_tag="medium",
    )

    # Ashaar tasks
    run_ashaar_rewrite(
        MEDIUM_SIZES["ashaar_rewrite"],
        decode_mode,
        MEDIUM_OUT_DIR,
        mode_tag="medium",
    )

    run_ashaar_label_task(
        MEDIUM_SIZES["ashaar_theme"],
        decode_mode,
        MEDIUM_OUT_DIR,
        label_list=theme_labels,
        label_key="poem theme",
        mode_tag="medium",
        valid_indices=theme_valid_indices,   # 👈 NEW
    )

    run_ashaar_label_task(
        MEDIUM_SIZES["ashaar_era"],
        decode_mode,
        MEDIUM_OUT_DIR,
        label_list=era_labels,
        label_key="poet era",
        mode_tag="medium",
        valid_indices=era_valid_indices,     # 👈 NEW
    )


In [16]:
# %%
# =========================
#  FULL run suite (optional, AFTER medium is clean)
# =========================
# When you're happy with the medium CSVs and metrics,
# you can use these helpers to run a larger/full evaluation.
# You can tweak FULL_SIZES above (None means "use full dataset").

def run_full_suite_for_decode(decode_mode):
    print("\n" + "="*60)
    print(f"Running FULL suite | model = {MODEL_TAG} | decode_mode = {decode_mode}")
    print("="*60)

    # Figure out sizes (None -> full length)
    def full_n(task_key, total):
        n = FULL_SIZES.get(task_key, None)
        return total if n is None else min(n, total)

    # Al-Ghafa
    arc_df = alghafa_dfs["alghafa_arc"]
    mmlu_df = alghafa_dfs["alghafa_mmlu"]
    boolq_df = alghafa_dfs["alghafa_boolq"]
    piqa_df = alghafa_dfs["alghafa_piqa"]

    run_alghafa_arc(
        arc_df,
        full_n("alghafa_arc", arc_df.shape[0]),
        decode_mode,
        FULL_OUT_DIR,
        mode_tag="full",
    )

    run_alghafa_mmlu(
        mmlu_df,
        full_n("alghafa_mmlu", mmlu_df.shape[0]),
        decode_mode,
        FULL_OUT_DIR,
        mode_tag="full",
    )

    run_alghafa_boolq(
        boolq_df,
        full_n("alghafa_boolq", boolq_df.shape[0]),
        decode_mode,
        FULL_OUT_DIR,
        mode_tag="full",
    )

    run_alghafa_piqa(
        piqa_df,
        full_n("alghafa_piqa", piqa_df.shape[0]),
        decode_mode,
        FULL_OUT_DIR,
        mode_tag="full",
    )

    # Ashaar
    n_total_ashaar = len(ashaar_ds)

    run_ashaar_rewrite(
        full_n("ashaar_rewrite", n_total_ashaar),
        decode_mode,
        FULL_OUT_DIR,
        mode_tag="full",
    )

    run_ashaar_label_task(
        full_n("ashaar_theme", n_total_ashaar),
        decode_mode,
        FULL_OUT_DIR,
        label_list=theme_labels,
        label_key="poem theme",
        mode_tag="full",
    )

    run_ashaar_label_task(
        full_n("ashaar_era", n_total_ashaar),
        decode_mode,
        FULL_OUT_DIR,
        label_list=era_labels,
        label_key="poet era",
        mode_tag="full",
    )


In [17]:
# %%
# 🔥 RUN THIS CELL FOR THE MEDIUM SANITY RUN 🔥
# It will:
#

In [18]:
# %%
# ============================
# Aggregate MEDIUM CSVs → one JSON
# ============================
import json
from pathlib import Path
import pandas as pd

MEDIUM_DIR = Path("results_medium")
OUT_JSON   = MEDIUM_DIR / "summary_medium.json"

agg = {}

for csv_path in sorted(MEDIUM_DIR.glob("*.csv")):
    name = csv_path.stem  # e.g. alghafa_arc_medium_Yehia-7B-preview_det
    parts = name.split("_")

    # Very light parsing, robust to small naming changes
    source      = parts[0] if len(parts) > 0 else "unknown"      # alghafa / ashaar
    task        = parts[1] if len(parts) > 1 else "unknown"      # arc / mmlu / boolq / piqa / rewrite / theme / era
    mode_tag    = parts[2] if len(parts) > 2 else ""             # medium
    model_tag   = parts[3] if len(parts) > 3 else "MODEL"
    decode_mode = parts[4] if len(parts) > 4 else "decode"

    df = pd.read_csv(csv_path)

    # --- basic metrics ---
    metrics = {}
    if "correct" in df.columns:
        metrics["accuracy"] = float(df["correct"].mean())
    if "is_correct" in df.columns:
        metrics["accuracy"] = float(df["is_correct"].mean())
    if "exact_match" in df.columns:
        metrics["exact_match"] = float(df["exact_match"].mean())
    if "char_match_ratio" in df.columns:
        metrics["char_match_ratio"] = float(df["char_match_ratio"].mean())

    entry = {
        "file": str(csv_path),
        "source": source,
        "task": task,
        "mode_tag": mode_tag,
        "decode_mode": decode_mode,
        "n_rows": int(len(df)),
        "metrics": metrics,
        "rows": df.to_dict(orient="records"),  # full info for later inspection
    }

    model_block = agg.setdefault(model_tag, {})
    decode_block = model_block.setdefault(decode_mode, {})
    task_key = f"{source}_{task}"
    decode_block[task_key] = entry

# Save JSON
with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(agg, f, ensure_ascii=False, indent=2)

print(f"Saved medium summary JSON to: {OUT_JSON}")
print("Models / decode modes found:")
for model_tag, by_decode in agg.items():
    for decode_mode in by_decode.keys():
        print("  -", model_tag, "|", decode_mode)


Saved medium summary JSON to: results_medium/summary_medium.json
Models / decode modes found:
  - Hala-9B | det
  - Hala-9B | samp
  - Yehia-7B-preview | det
  - Yehia-7B-preview | samp


In [19]:
# %%
# ⚠️ DON'T RUN THIS UNTIL WE ARE HAPPY WITH MEDIUM RESULTS ⚠️

!pip install accelerate

for model_id in [HALA_ID, YEHIA_ID]:
     load_model(model_id)
     for decode_mode in DECODE_MODES:
         run_full_suite_for_decode(decode_mode)



Loading model: hammh0a/Hala-9B


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

...

Ashaar rewrite full (samp) | EM=0.567, char_match=0.909
Saved: results_full/ashaar_rewrite_full_Yehia-7B-preview_samp.csv


Ashaar الموضوع (samp, full):   0%|          | 0/300 [00:00<?, ?it/s]

Ashaar theme full (samp) accuracy: 0.107
Saved: results_full/ashaar_theme_full_Yehia-7B-preview_samp.csv


Ashaar العصر (samp, full):   0%|          | 0/300 [00:00<?, ?it/s]

Ashaar era full (samp) accuracy: 0.133
Saved: results_full/ashaar_era_full_Yehia-7B-preview_samp.csv
